# Token Classification (NER)

> Named entity recognition and other token-level tasks using `TokenClassificationLearner`.

Start with a quick NER flow, then move to custom-token datasets and advanced inference options.

## Notebook Metadata

`nbdev` export target for this notebook.

In [ ]:
#| default_exp text.token_classification


## Setup

Import the learner and helper libraries.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import pandas as pd

from blurr.text.token_classification import TokenClassificationLearner


## Quick Start

Fine-tune a compact model on CoNLL-2003 and run one NER prediction.

In [ ]:
learn = TokenClassificationLearner.from_hf(
    'conll2003',
    model_name='prajjwal1/bert-tiny',
    tokens_col='tokens',
    labels_col='ner_tags',
    max_samples=200,
    max_length=128,
    bs=16,
)

learn.fit_one_cycle(1, 2e-5)
quick_entities = learn.blurr_predict_tokens('Hugging Face is based in New York City.')
quick_entities[0]


**Expected output**

A list of entity spans, each with fields like `entity_group`, `word`, `start`, `end`, and `score`.

## Aggregation Strategies

Different aggregation strategies control how subword predictions are merged into final entity spans.

In [ ]:
for strategy in ['simple', 'first', 'max', 'average']:
    entities = learn.blurr_predict_tokens(
        'Apple Inc. was founded by Steve Jobs in Cupertino, California.',
        aggregation_strategy=strategy,
    )[0]
    print(f'\n--- {strategy} ---')
    for e in entities:
        print(e)


## From a DataFrame

For custom datasets, provide tokenized inputs and aligned token labels per row.

In [ ]:
custom_df = pd.DataFrame({
    'tokens': [
        ['John', 'lives', 'in', 'New', 'York', '.'],
        ['Mary', 'works', 'at', 'Google', '.'],
        ['Paris', 'is', 'in', 'France', '.'],
    ],
    'labels': [
        ['B-PER', 'O', 'O', 'B-LOC', 'I-LOC', 'O'],
        ['B-PER', 'O', 'O', 'B-ORG', 'O'],
        ['B-LOC', 'O', 'O', 'B-LOC', 'O'],
    ],
})

learn_df = TokenClassificationLearner.from_df(
    custom_df,
    model_name='prajjwal1/bert-tiny',
    tokens_col='tokens',
    labels_col='labels',
    bs=2,
    max_length=64,
)


## Advanced Notes

- Subword alignment is handled automatically.
- Special tokens and padded positions are ignored via `ignore_token_id` (`-100`).
- If `seqeval` is installed, entity-level precision/recall/F1 are logged during validation.

## API Reference

### Main class
- `TokenClassificationLearner`

### Construction methods
- `from_hf(dataset, model_name, ...)`
- `from_df(df, model_name, ...)`

### Inference
- `blurr_predict_tokens(texts, aggregation_strategy='simple')`

### Key options
- `label_all_tokens` controls whether subword continuations receive labels
- `label_names` allows explicit label vocabulary control
